# 03 — Эксперименты и Ablation Study

Цель:
- Подбор гиперпараметров (GridSearchCV / RandomizedSearchCV)
- Ablation study: влияние engineered features
- Уменьшение размерности (PCA)
- Итоговая таблица экспериментов
- Обоснование выбора финальной модели

## 0. Настройка окружения

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f'Корень проекта: {project_root}')

In [ ]:
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.model_selection import RandomizedSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, roc_auc_score, classification_report
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from src.modeling import (
    load_splits,
    build_preprocessor,
    evaluate,
    save_model,
    ORIGINAL_COLS,
)

warnings.filterwarnings('ignore')

## 1. Загрузка данных

In [ ]:
X_train, y_train, X_val, y_val, X_test, y_test = load_splits(project_root)

# Признаки без engineered
original_cols = [c for c in ORIGINAL_COLS if c in X_train.columns]
X_train_base = X_train[original_cols]
X_val_base = X_val[original_cols]
X_test_base = X_test[original_cols]

## 2. Ablation Study: влияние engineered features

Сравниваем LightGBM с исходными признаками и с добавленными engineered.

In [ ]:
ablation_results = []

for label, X_tr, X_vl in [
    ('LightGBM (original only)', X_train_base, X_val_base),
    ('LightGBM (original + engineered)', X_train, X_val),
]:
    pipe = Pipeline([
        ('pre', build_preprocessor(X_tr)),
        ('clf', LGBMClassifier(random_state=42, verbose=-1)),
    ])
    pipe.fit(X_tr, y_train)
    metrics = evaluate(pipe, X_vl, y_val, label=label)
    metrics['experiment'] = label
    ablation_results.append(metrics)

ablation_df = pd.DataFrame(ablation_results).set_index('experiment')
display(ablation_df)

## 3. Уменьшение размерности (PCA)

Проверяем, помогает ли PCA сохранить качество при меньшем числе компонент.

In [ ]:
pca_results = []
n_components_list = [5, 10, 15, 20, None]  # None = без PCA

for n_comp in n_components_list:
    steps = [('pre', build_preprocessor(X_train))]
    if n_comp is not None:
        steps.append(('pca', PCA(n_components=n_comp, random_state=42)))
    steps.append(('clf', LGBMClassifier(random_state=42, verbose=-1)))

    pipe = Pipeline(steps)
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_val)
    y_prob = pipe.predict_proba(X_val)[:, 1]

    label = f'PCA n={n_comp}' if n_comp else 'No PCA'
    pca_results.append({
        'n_components': n_comp if n_comp else 'all',
        'f1': f1_score(y_val, y_pred),
        'roc_auc': roc_auc_score(y_val, y_prob),
    })
    print(f"{label}: F1={pca_results[-1]['f1']:.4f}, AUC={pca_results[-1]['roc_auc']:.4f}")

pca_df = pd.DataFrame(pca_results)
display(pca_df)

# Визуализация
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(pca_df['n_components'].astype(str), pca_df['f1'], marker='o', label='F1')
ax.plot(pca_df['n_components'].astype(str), pca_df['roc_auc'], marker='s', label='ROC-AUC')
ax.set_title('LightGBM: влияние числа компонент PCA')
ax.set_xlabel('Число компонент')
ax.set_ylabel('Метрика')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Подбор гиперпараметров (RandomizedSearchCV)

Тюним LightGBM как лучшую одиночную модель.

In [ ]:
param_dist = {
    'clf__n_estimators': [100, 200, 300, 500],
    'clf__max_depth': [-1, 5, 10, 15],
    'clf__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'clf__num_leaves': [31, 63, 127],
    'clf__min_child_samples': [10, 20, 50],
    'clf__subsample': [0.7, 0.8, 1.0],
    'clf__colsample_bytree': [0.7, 0.8, 1.0],
}

base_pipe = Pipeline([
    ('pre', build_preprocessor(X_train)),
    ('clf', LGBMClassifier(random_state=42, verbose=-1)),
])

search = RandomizedSearchCV(
    base_pipe,
    param_distributions=param_dist,
    n_iter=30,
    scoring='f1',
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1,
)
search.fit(X_train, y_train)

print(f"\nЛучшие параметры: {search.best_params_}")
print(f"Лучший CV F1: {search.best_score_:.4f}")

In [ ]:
best_lgbm = search.best_estimator_
tuned_metrics = evaluate(best_lgbm, X_val, y_val, label='LightGBM (tuned)')

## 5. Итоговая таблица экспериментов

In [ ]:
from src.modeling import train_baseline, train_all_models, train_ensemble

# Baseline
baseline_pipe, baseline_metrics = train_baseline(X_train, y_train, X_val, y_val)

# Все модели без тюнинга
pipelines, results_df = train_all_models(X_train, y_train, X_val, y_val)

# Ансамбль
ensemble, ensemble_metrics = train_ensemble(pipelines, X_train, y_train, X_val, y_val)

# Сводная таблица
all_experiments = [
    {'Эксперимент': 'Baseline (Logistic Regression)', **baseline_metrics},
]
for model_name, row in results_df.iterrows():
    all_experiments.append({'Эксперимент': model_name, **row.to_dict()})

all_experiments.append({'Эксперимент': 'Ensemble (Soft Voting)', **ensemble_metrics})
all_experiments.append({'Эксперимент': 'LightGBM (tuned)', **tuned_metrics})

exp_df = pd.DataFrame(all_experiments).set_index('Эксперимент')
exp_df = exp_df.sort_values('f1', ascending=False)

print('Итоговая таблица экспериментов (Val):')
display(exp_df.style.highlight_max(color='lightgreen'))

## 6. Обоснование выбора финальной модели и тестовая оценка

In [ ]:
best_exp = exp_df['f1'].idxmax()
print(f"Финальная модель: {best_exp}")
print(
    "Обоснование: выбираем модель с наивысшим F1 на валидационной выборке. "
    "F1 приоритетен, так как классы несбалансированы и важно минимизировать как "
    "ложноположительные, так и ложноотрицательные предсказания."
)

# Определяем финальную модель
if 'tuned' in best_exp.lower():
    final_model = best_lgbm
elif 'ensemble' in best_exp.lower():
    final_model = ensemble
elif best_exp in pipelines:
    final_model = pipelines[best_exp]
else:
    final_model = baseline_pipe

print('\n--- Финальная оценка на TEST ---')
test_metrics = evaluate(final_model, X_test, y_test, label=f'Final: {best_exp}')
print(f"Test F1: {test_metrics['f1']:.4f} | Test ROC-AUC: {test_metrics['roc_auc']:.4f}")

safe_name = best_exp.replace(' ', '_').replace('(', '').replace(')', '').lower()
save_model(final_model, f'final_{safe_name}', project_root)